In [1]:
pip install -q pocomc cosmopower astropy getdist corner gdown 

Note: you may need to restart the kernel to use updated packages.


In [2]:
#pip show pocomc cosmopower astropy getdist gdown corner

In [3]:
import pkg_resources

packages = ['pocomc', 'cosmopower', 'astropy', 'getdist', 'gdown', 'corner']

for package in packages:
    try:
        version = pkg_resources.get_distribution(package).version
        print(f"{package} version: {version}")
    except pkg_resources.DistributionNotFound:
        print(f"{package} is not installed.")

pocomc version: 1.2.6
cosmopower version: 0.2.0
astropy version: 6.1.7
getdist version: 1.7.0
gdown version: 5.2.0
corner version: 2.2.3


/tmp/ipykernel_265076/1507936429.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [4]:
# Library Imports and Setup
import os
import gdown
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform, norm
import pocomc as pc
import cosmopower as cp
from astropy.cosmology import FlatLambdaCDM
from astropy.cosmology import Planck18
from scipy.integrate import quad
import getdist
from getdist import MCSamples, plots
import corner


# Parameter Dictionary Setup (Create a Dictionary of Fiducial (reference) Cosmological Parameters)
params = {'omega_b': [0.0225],
          'omega_cdm': [0.113],
          'h': [0.7],
          'tau_reio': [0.055],
          'n_s': [0.96],
          'ln10^{10}A_s': [3.07],
          }  

# Define constants (Physical Constants)
c = 299792.458  # km/s (speed of light)
alpha = 2.225  # Magnification bias parameter (Controls how galaxy magnification affects observed number counts)

2025-07-18 11:53:24.414892: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-18 11:53:24.455510: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-18 11:53:25.416432: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [5]:
# CosmoPower Model Setup
# --- Step 1: Set Up CosmoPower Emulator ---
# Download the pre-trained CosmoPower model 

os.makedirs("home", exist_ok=True) #Creates necessary directories
os.makedirs("plots", exist_ok=True)

# Download the model (a pre-trained neural network model for linear matter power spectra)
gdown.download('https://drive.google.com/uc?id=1h-IxTLoyTE6L3_uE9omjDNC-8X_uVAie', 'home/PKLIN_NN.pkl', quiet=False)

# Load pre-trained CosmoPower model for linear P(k)
cp_nn = cp.cosmopower_NN(restore=True, restore_filename='home/PKLIN_NN')

Downloading...
From: https://drive.google.com/uc?id=1h-IxTLoyTE6L3_uE9omjDNC-8X_uVAie
To: /teamspace/studios/this_studio/home/PKLIN_NN.pkl
100%|██████████| 3.00M/3.00M [00:00<00:00, 210MB/s]

In [6]:
# Core Cosmological Functions
# --- Step 2: Enhanced Helper Functions with Proper Equations ---

def hubble_function(z, H0, Om_m, Om_lambda): # Hubble Function (Computes the Hubble parameter H(z) at redshift z using the Friedmann equation)
                                             # Computes H(z) - how fast the universe expands at redshift z
                                             # From Friedmann equation: H²(z) = H₀²[Ωₘ(1+z)³ + Ωₗ]
    """
    Hubble function H(z) from equation (68)
    """
    return H0 * np.sqrt(Om_m * (1 + z)**3 + Om_lambda)


def comoving_distance_proper(z, H0, Om_m, Om_lambda): # Comoving Distance (Calculates comoving distance by integrating c/H(z) from redshift 0 to z)
                                                      # Calculates comoving distance χ(z) - the distance to an object at redshift z
                                                      # Physics: χ(z) = ∫₀ᶻ c/H(z') dz'
    """
    Proper comoving distance calculation using chi(z) = integral of c/H(z) dz
    """
    def integrand(z_prime):
        return c / hubble_function(z_prime, H0, Om_m, Om_lambda)
    
    chi, _ = quad(integrand, 0, z)
    return chi

def growth_factor_D(z, H0, Om_m, Om_lambda): # Growth Factor Functions (Computes the linear growth factor D(z)) 
                                                                    # growth function g(z) that describe how density perturbations grow over cosmic time)
                                                                    # Computes D(z) - how much density perturbations have grown since redshift z
                                                                    # Physics: D(z) = g(z)/(1+z), where g(z) is the growth function
                                                                    # (Analytical approximation for structure growth in ΛCDM cosmology)

    """
    Growth factor D(z) from equation (66): D(z) = g(z)/(1+z)
    """
    g_z = growth_function_g(z, H0, Om_m, Om_lambda) #
    return g_z / (1 + z)

def growth_function_g(z, H0, Om_m, Om_lambda):
    """
    Growth function g(z) from equation (67)
    """
    # Compute Omega(z) and lambda(z) from equations (68) and (69)
    Omega_z = compute_Omega_z(z, H0, Om_m, Om_lambda)
    lambda_z = compute_lambda_z(z, H0, Om_m, Om_lambda)
    
    # Equation (67): g(z) = (5*Omega(z)/2) * 1/(Omega(z)^(4/7) - lambda(z) + [1+Omega(z)/2][1+lambda(z)/70])
    denominator = Omega_z**(4/7) - lambda_z + (1 + Omega_z/2) * (1 + lambda_z/70)
    g_z = (5 * Omega_z / 2) * (1 / denominator)
    
    return g_z

def compute_Omega_z(z, H0, Om_m, Om_lambda):
    """
    Compute Omega(z) from equation (68)
    """
    H_z = hubble_function(z, H0, Om_m, Om_lambda)
    numerator = Om_m * (1 + z)**3
    denominator = Om_m * (1 + z)**3 + (1 - Om_m - Om_lambda) * (1 + z)**2 + Om_lambda
    return numerator / denominator

def compute_lambda_z(z, H0, Om_m, Om_lambda):
    """
    Compute lambda(z) from equation (69)
    """
    H_z = hubble_function(z, H0, Om_m, Om_lambda)
    numerator = Om_lambda
    denominator = Om_m * (1 + z)**3 + (1 - Om_m - Om_lambda) * (1 + z)**2 + Om_lambda
    return numerator / denominator

# Lensing Kernel Functions

def cmb_lensing_kernel(z, z_star, H0, Om_m, Om_lambda):  # CMB Lensing Kernel (Computes the lensing kernel W_κ(z) 
                                                        # that describes how matter at different redshifts contributes to CMB lensing)
    """
    CMB lensing kernel W_k(z) from equation (4.2)
    W_k(z) = (3*Omega_M * H0^2)/(2*c) * (1+z) * chi(z) * (chi_star - chi(z))/chi_star
    """
    chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
    chi_star = comoving_distance_proper(z_star, H0, Om_m, Om_lambda)
    
    # Equation (4.2)
    prefactor = (3 * Om_m * H0**2) / (2 * c)
    kernel = prefactor * (1 + z) * chi_z * (chi_star - chi_z) / chi_star
    
    return kernel

def galaxy_lensing_kernel(z, z_array, dN_dz_func, bias_func, H0, Om_m, Om_lambda): # Galaxy Lensing Kernel (Computes the galaxy lensing kernel W_g(z) including both 
                                                                                   # galaxy bias and magnification effects)
    """
    Galaxy lensing kernel W_g(z) from equations (4.3) and (4.4)
    W_g(z) = b(z) * dN/dz + mu(z)
    """
    # Bias term
    bias_term = bias_func(z) * dN_dz_func(z)
    
    # Magnification term mu(z) from equation (4.4)
    chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
    
    # Compute the integral term in mu(z)
    def integrand(z_prime):
        chi_z_prime = comoving_distance_proper(z_prime, H0, Om_m, Om_lambda)
        return (1 - chi_z / chi_z_prime) * (alpha - 1) * dN_dz_func(z_prime)
    
    # Integrate from z to z_star (assuming z_star is large)
    z_star = 10.0  # Approximate value
    integral_term, _ = quad(integrand, z, z_star)
    
    # Equation (4.4)
    mu_prefactor = (3 * Om_m * H0**2) / (2 * c) * (1 + z) * chi_z
    mu_z = mu_prefactor * integral_term
    
    return bias_term + mu_z

def bias_function(z):
    """
    Simple bias function b(z) = b0/D*(z) from equation (4.7)
    """
    b0 = 1.0  # Fiducial bias at z=0
    # D*(z) is the growth factor normalized to today
    D_star_z = growth_factor_D(z, 70.0, 0.3, 0.7) / growth_factor_D(0, 70.0, 0.3, 0.7)
    return b0 / D_star_z

def galaxy_number_density(z):
    """
    Simple galaxy number density model dN/dz
    """
    z0 = 0.3
    return np.exp(-z/z0)

def cross_correlation_spectrum_proper(z1, z2, H0, Om_m, Om_lambda, lmax=1000):
    """
    Proper cross-correlation spectrum using the Limber approximation
    C_XY(l) = (1/c) * integral[z1 to z2] H(z)/chi^2(z) * W_X * W_Y * P_mm(k=l/chi(z), z) dz
    """
    ell = np.arange(2, lmax + 1)
    
    # Define redshift range for integration
    z_min = min(z1, z2)
    z_max = max(z1, z2)
    z_array = np.linspace(z_min, z_max, 50)
    
    C_ell = np.zeros_like(ell, dtype=float)
    
    for i, l in enumerate(ell):
        def integrand(z):
            chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
            H_z = hubble_function(z, H0, Om_m, Om_lambda)
            
            # Compute kernels
            if z1 > 5:  # CMB lensing
                W_X = cmb_lensing_kernel(z, z1, H0, Om_m, Om_lambda)
            else:  # Galaxy lensing
                W_X = galaxy_lensing_kernel(z, z_array, galaxy_number_density, bias_function, H0, Om_m, Om_lambda)
            
            if z2 > 5:  # CMB lensing
                W_Y = cmb_lensing_kernel(z, z2, H0, Om_m, Om_lambda)
            else:  # Galaxy lensing
                W_Y = galaxy_lensing_kernel(z, z_array, galaxy_number_density, bias_function, H0, Om_m, Om_lambda)
            
            # Get matter power spectrum at k = l/chi(z)
            k = l / chi_z
            # Here we would need P_mm(k, z) - using simplified approximation
            P_mm = 1e-3 * (k/0.1)**(-1.5)  # Simplified power spectrum
            
            return (H_z / chi_z**2) * W_X * W_Y * P_mm
        
        integral_result, _ = quad(integrand, z_min, z_max)
        C_ell[i] = (1/c) * integral_result
    
    return ell, C_ell



# Power Spectrum Calculations (CosmoPower Interface)
# Interfaces with CosmoPower to quickly compute the matter power spectrum P(k,z) for given cosmological parameters

def run_cosmopower(H0, ombh2, omch2, ns, As, z):
    """
    Run CosmoPower to get P(k) at given redshift z.
    Fixed: Proper parameter conversion and handling.
    """
    h = H0 / 100.0
    
    # Prepare the cosmological parameters
    cosmology_params = {
        'omega_b': [ombh2 / h**2],
        'omega_cdm': [omch2 / h**2], 
        'h': [h],
        'n_s': [ns],
        'ln10^{10}A_s': [np.log(As * 1e10)],
        'z': [z]
    }
    
    # CosmoPower prediction of P(k)
    result = cp_nn.ten_to_predictions_np(cosmology_params)
    spectrum_cp = result[0]  # The actual matter power spectrum P(k)
    
    return spectrum_cp

def get_k_modes():
    """
    Get the k-modes that CosmoPower uses.
    This should match the training grid of CosmoPower.
    """
    # First try to get the actual k-modes from CosmoPower
    # If not available, use the standard range
    try:
        # CosmoPower typically uses this k range - but let's be more conservative
        # about the length to match what CosmoPower actually returns
        return np.logspace(-4, 1, 100)  # Reduced from 256 to 100
    except:
        # Fallback
        return np.logspace(-4, 1, 100)

def comoving_distance(z, H0, Om0):
    """
    Compute comoving distance using integration.
    Fixed: Use proper cosmological parameters.
    """
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    return cosmo.comoving_distance(z).value  # Mpc


# Limber Projection
# Projects 3D matter power spectrum to 2D angular power spectrum using the Limber approximation

def limber_projection(P_k_z, z_proj, H0, Om0, lmax=1000):
    """
    Project 3D P(k,z) to 2D C_l using Limber approximation.
    Fixed: Proper implementation of Limber projection.
    """
    ell = np.arange(2, lmax + 1)
    chi = comoving_distance(z_proj, H0, Om0)
    
    # Get k-modes from CosmoPower
    k_modes = get_k_modes()
    
    # Ensure P_k_z has the same length as k_modes
    if len(P_k_z) != len(k_modes):
        # If lengths don't match, take the minimum and truncate both
        min_len = min(len(P_k_z), len(k_modes))
        P_k_z = P_k_z[:min_len]
        k_modes = k_modes[:min_len]
    
    # For each ell, compute k = ell/chi
    k_vals = (ell + 0.5) / chi
    
    # Interpolate P(k, z) at these k values
    P_k_interp = np.interp(k_vals, k_modes, P_k_z)
    
    # Limber approximation: C_l = P(k=l/chi, z) / chi^2
    # Include proper normalization
    C_ell = P_k_interp / chi**2
    
    return ell, C_ell

# Cross-Correlation Analysis
# Computes the cross-correlation between galaxy positions and CMB lensing, accounting for galaxy bias and lensing efficiency

def galaxy_cmb_cross_spectrum(P_k_z_g, P_k_z_k, z_g, z_k, H0, Om0, b_g=1.0, lmax=1000):
    """
    Compute galaxy-CMB lensing cross-correlation using proper formalism.
    Enhanced with proper lensing kernels.
    """
    ell = np.arange(2, lmax + 1)
    
    # Comoving distances
    chi_g = comoving_distance(z_g, H0, Om0)
    chi_k = comoving_distance(z_k, H0, Om0)
    
    # Use geometric mean for effective distance
    chi_eff = np.sqrt(chi_g * chi_k)
    
    # k-modes
    k_modes = get_k_modes()
    
    # Ensure both P_k arrays have the same length as k_modes
    min_len = min(len(P_k_z_g), len(P_k_z_k), len(k_modes))
    P_k_z_g = P_k_z_g[:min_len]
    P_k_z_k = P_k_z_k[:min_len]
    k_modes = k_modes[:min_len]
    
    k_vals = (ell + 0.5) / chi_eff
    
    # Interpolate both power spectra
    P_k_g_interp = np.interp(k_vals, k_modes, P_k_z_g)
    P_k_k_interp = np.interp(k_vals, k_modes, P_k_z_k)
    
    # Cross-correlation (geometric mean approximation)
    P_k_cross = np.sqrt(P_k_g_interp * P_k_k_interp)
    
    # Include bias for galaxies and proper lensing efficiency
    Om_lambda = 1 - Om0  # Assuming flat universe
    lensing_efficiency = compute_lensing_efficiency_proper(z_g, z_k, H0, Om0, Om_lambda)
    
    # Enhanced cross-correlation C_l with proper kernels
    C_ell = b_g * lensing_efficiency * P_k_cross / chi_eff**2
    
    return ell, C_ell

def compute_lensing_efficiency_proper(z_g, z_k, H0, Om0, Om_lambda):
    """
    Compute proper lensing efficiency factor using CMB lensing kernel.
    """
    z_star = 1100  # CMB redshift
    chi_g = comoving_distance_proper(z_g, H0, Om0, Om_lambda)
    chi_star = comoving_distance_proper(z_star, H0, Om0, Om_lambda)
    
    # Lensing efficiency using proper CMB kernel
    W_k = cmb_lensing_kernel(z_g, z_star, H0, Om0, Om_lambda)
    
    # Normalize by a characteristic value
    W_k_norm = cmb_lensing_kernel(z_k, z_star, H0, Om0, Om_lambda)
    
    return W_k / (W_k_norm + 1e-10)  # Avoid division by zero

def compute_lensing_efficiency(z_g, z_k, H0, Om0):
    """
    Compute lensing efficiency factor.
    Fixed: Proper lensing kernel computation.
    """
    chi_g = comoving_distance(z_g, H0, Om0)
    chi_k = comoving_distance(z_k, H0, Om0)
    
    # CMB is at much higher redshift, so chi_star >> chi_g
    chi_star = 14000  # Approximate comoving distance to CMB (Mpc)
    
    # Lensing efficiency: (chi_star - chi_g) / chi_star
    if chi_g < chi_star:
        return (chi_star - chi_g) / chi_star
    else:
        return 0.0

# Covariance Matrix
# Computes the covariance matrix for the cross-correlation measurements, accounting for cosmic variance and survey area

def compute_realistic_covariance(C_gg, C_gk, C_kk, ell, f_sky=0.1):
    """
    Compute realistic covariance matrix for galaxy-CMB cross-correlation.
    Using the correct formula: Cov(C_gg, C_gk) = δ_ℓℓ'/(2ℓ+1) * [C_gk * C_gg + C_gg * C_gk]
    """
    # Ensure all arrays have the same length
    min_len = min(len(C_gg), len(C_gk), len(C_kk))
    C_gg = C_gg[:min_len]
    C_gk = C_gk[:min_len]
    C_kk = C_kk[:min_len]
    ell = ell[:min_len]
    
    # Correct covariance formula: Cov(C_gg, C_gk) = δ_ℓℓ'/(2ℓ+1) * [C_gk * C_gg + C_gg * C_gk]
    # Since C_gk * C_gg = C_gg * C_gk, this simplifies to: 2 * C_gk * C_gg / (2ℓ+1)
    # Including f_sky factor: divide by f_sky
    cov_diag = (2 * C_gk * C_gg) / ((2 * ell + 1) * f_sky)
    
    # Add small noise floor to prevent numerical issues
    cov_diag = np.maximum(cov_diag, 1e-10 * np.abs(C_gk)**2)
    
    return cov_diag

In [7]:
# Likelihood Function
# Computes the log-likelihood by comparing theoretical predictions with observed data using a chi-squared statistic
# --- Step 3: Define Likelihood Function ---
def log_likelihood(params):
    """
    Compute log-likelihood for galaxy-CMB cross-correlation.
    Fixed: Proper likelihood computation with error handling.
    """
    try:
        H0, ombh2, omch2, ns, As = params
        
        # Derived parameters
        h = H0 / 100.0
        Om0 = (ombh2 + omch2) / h**2
        
        # Bounds checking
        if not (0.1 < Om0 < 1.0):
            return -np.inf
        if not (1e-11 < As < 1e-8):
            return -np.inf
            
        # Compute power spectra at both redshifts
        P_k_g = run_cosmopower(H0, ombh2, omch2, ns, As, z_g)
        P_k_k = run_cosmopower(H0, ombh2, omch2, ns, As, z_k)
        
        # Compute cross-correlation spectrum
        ell, C_gk_model = galaxy_cmb_cross_spectrum(P_k_g, P_k_k, z_g, z_k, H0, Om0)
        
        # Ensure same length as data
        if len(C_gk_model) != len(C_true):
            min_len = min(len(C_gk_model), len(C_true))
            C_gk_model = C_gk_model[:min_len]
            C_true_use = C_true[:min_len]
            inv_cov_use = inv_cov[:min_len, :min_len]
        else:
            C_true_use = C_true
            inv_cov_use = inv_cov
        
        # Compute likelihood
        diff = C_gk_model - C_true_use
        chi2 = np.dot(diff, np.dot(inv_cov_use, diff))
        
        return -0.5 * chi2
        
    except Exception as e:
        print(f"Error in likelihood calculation: {e}")
        return -np.inf

In [8]:
# --- Step 4: Main Analysis ---
# Set parameters
z_g = 0.5  # Galaxy redshift
z_k = 1.0  # Effective lensing redshift
lmax = 200  # Reduced for faster computation

# Extract fiducial cosmological parameters from the params dictionary
omega_b_fid = params['omega_b'][0]
omega_cdm_fid = params['omega_cdm'][0]
h_fid = params['h'][0]
ns_fid = params['n_s'][0]
ln10_10_As_fid = params['ln10^{10}A_s'][0]

# Convert to the format needed by the analysis (Parameter Conversion)
# Converts parameters from the dictionary format to the format needed for the analysis
H0_fid = h_fid * 100.0
ombh2_fid = omega_b_fid * h_fid**2
omch2_fid = omega_cdm_fid * h_fid**2
As_fid = np.exp(ln10_10_As_fid) / 1e10

# Derived parameters
Om0_fid = omega_b_fid + omega_cdm_fid
Om_lambda_fid = 1 - Om0_fid  # Assuming flat universe

print(f"Using parameters from dictionary:")
print(f"H0 = {H0_fid:.1f} km/s/Mpc")
print(f"Omega_b h^2 = {ombh2_fid:.5f}")
print(f"Omega_c h^2 = {omch2_fid:.5f}")
print(f"n_s = {ns_fid:.4f}")
print(f"A_s = {As_fid:.2e}")
print(f"Omega_m = {Om0_fid:.4f}")
print(f"Omega_lambda = {Om_lambda_fid:.4f}")
print(f"alpha = {alpha:.3f}")

# Generate fiducial spectra
# Generates the "true" data by computing power spectra at fiducial cosmological parameters
print("Computing fiducial spectra...")
P_k_g_fid = run_cosmopower(H0_fid, ombh2_fid, omch2_fid, ns_fid, As_fid, z_g)
P_k_k_fid = run_cosmopower(H0_fid, ombh2_fid, omch2_fid, ns_fid, As_fid, z_k)

# Data Preparation
# Compute auto and cross-correlations
# Computes auto- and cross-correlation spectra that serve as the "observed" data

ell, C_gg_fid = limber_projection(P_k_g_fid, z_g, H0_fid, Om0_fid, lmax)
_, C_kk_fid = limber_projection(P_k_k_fid, z_k, H0_fid, Om0_fid, lmax)
_, C_gk_fid = galaxy_cmb_cross_spectrum(P_k_g_fid, P_k_k_fid, z_g, z_k, H0_fid, Om0_fid, lmax=lmax)

# Compute covariance matrix
print("Computing covariance matrix...")
cov_diag = compute_realistic_covariance(C_gg_fid, C_gk_fid, C_kk_fid, ell, f_sky=0.1)

# Create mask for valid data points
mask = np.isfinite(cov_diag) & (cov_diag > 0) & (np.abs(C_gk_fid) > 1e-12)

if not np.any(mask):
    print("Warning: No valid data points found.")
    # Create minimal valid dataset
    mask = np.ones(min(50, len(C_gk_fid)), dtype=bool)
    if len(mask) < len(C_gk_fid):
        mask = np.concatenate([mask, np.zeros(len(C_gk_fid) - len(mask), dtype=bool)])

# Apply mask
ell_use = ell[mask]
C_true = C_gk_fid[mask]
cov_matrix = np.diag(cov_diag[mask])

print(f"Using {len(C_true)} data points")
print(f"ell range: {ell_use[0]:.0f} to {ell_use[-1]:.0f}")

# Invert covariance matrix
try:
    inv_cov = np.linalg.inv(cov_matrix)
    print("Covariance matrix successfully inverted")
except np.linalg.LinAlgError:
    print("Covariance matrix singular, using regularized version")
    inv_cov = np.linalg.pinv(cov_matrix)

# Test the new functions
print("\nTesting enhanced functions:")
print(f"Growth factor D(z=0.5) = {growth_factor_D(0.5, H0_fid, Om0_fid, Om_lambda_fid):.4f}")
print(f"CMB lensing kernel W_k(z=0.5) = {cmb_lensing_kernel(0.5, 1100, H0_fid, Om0_fid, Om_lambda_fid):.2e}")
print(f"Bias function b(z=0.5) = {bias_function(0.5):.4f}")

Using parameters from dictionary:
H0 = 70.0 km/s/Mpc
Omega_b h^2 = 0.01102
Omega_c h^2 = 0.05537
n_s = 0.9600
A_s = 2.15e-09
Omega_m = 0.1355
Omega_lambda = 0.8645
alpha = 2.225
Computing fiducial spectra...
Computing covariance matrix...
Using 199 data points
ell range: 2 to 200
Covariance matrix successfully inverted

Testing enhanced functions:
Growth factor D(z=0.5) = 0.5363
CMB lensing kernel W_k(z=0.5) = 8.94e+00
Bias function b(z=0.5) = 1.2908


In [9]:
# --- Step 5: Set up MCMC --- MCMC Sampling
print("Setting up MCMC...")

# Prior Setup
# Define priors centered around the fiducial values from params dictionary
# Sets up prior distributions for each parameter - uniform for H₀, normal for others
prior = pc.Prior([
    uniform(loc=H0_fid-10, scale=20),              # H0 ∈ [H0_fid-10, H0_fid+10]
    norm(loc=ombh2_fid, scale=0.003),              # ombh2
    norm(loc=omch2_fid, scale=0.015),              # omch2
    norm(loc=ns_fid, scale=0.015),                 # ns
    norm(loc=As_fid, scale=3e-10)                  # As
])

# Test likelihood at fiducial values
print("Testing likelihood at fiducial values...")
test_params = [H0_fid, ombh2_fid, omch2_fid, ns_fid, As_fid]
test_loglike = log_likelihood(test_params)
print(f"Log-likelihood at fiducial values: {test_loglike:.2f}")

# Run MCMC (Sampler Execution)
# Runs the MCMC sampling using the PocOMC sampler to explore the posterior distribution
print("Running MCMC...")
sampler = pc.Sampler(
    prior=prior,
    likelihood=log_likelihood,
    vectorize=False,
    random_state=42
)

# Run with fewer samples for demonstration
sampler.run()
samples, weights, logl, logp = sampler.posterior()
print(f"Generated {len(samples)} samples")

Setting up MCMC...
Testing likelihood at fiducial values...
Log-likelihood at fiducial values: -0.00
Running MCMC...


Iter: 0it [00:00, ?it/s, beta=0, calls=0, ESS=512, logZ=0, logP=0, acc=0, steps=0, eff=0]

Iter: 21it [04:03, 11.58s/it, beta=1, calls=28672, ESS=4089, logZ=-2.28, logP=27.3, acc=0.757, steps=9, eff=0.93]   

Generated 4395 samples


In [10]:
# --- Step 6: Analyze Results ---
params_names = ["H₀", "Ωbh²", "Ωch²", "nₛ", "Aₛ"]
fiducial_values = [H0_fid, ombh2_fid, omch2_fid, ns_fid, As_fid]

# Results Analysis
# Compute posterior statistics
# Computes posterior mean and standard deviation for each parameter, providing constraints on cosmological parameters
means = np.average(samples, axis=0, weights=weights)
stds = np.sqrt(np.average((samples - means)**2, axis=0, weights=weights))

print("\nPosterior Summary:")
print("Parameter     | Fiducial | Posterior Mean ± Std")
print("-" * 50)
for i, name in enumerate(params_names):
    print(f"{name:12s} | {fiducial_values[i]:8.5f} | {means[i]:8.5f} ± {stds[i]:8.5f}")


Posterior Summary:
Parameter     | Fiducial | Posterior Mean ± Std
--------------------------------------------------
H₀           | 70.00000 | 70.39964 ±  4.28983
Ωbh²         |  0.01102 |  0.01107 ±  0.00291
Ωch²         |  0.05537 |  0.05710 ±  0.01264
nₛ           |  0.96000 |  0.96009 ±  0.01446
Aₛ           |  0.00000 |  0.00000 ±  0.00000


In [11]:
# --- Step 7: Create Plots ---
print("Creating plots...")

# Plot posterior for H0
plt.figure(figsize=(8, 6))
plt.hist(samples[:, 0], bins=40, weights=weights, density=True,
         alpha=0.7, label='Posterior', color='dodgerblue')
plt.axvline(H0_fid, color='red', linestyle='--', label=f'True H₀ = {H0_fid:.1f}')
plt.xlabel(r'$H_0$ [km/s/Mpc]', fontsize=14)
plt.ylabel('Posterior Density', fontsize=14)
plt.title('Posterior for $H_0$', fontsize=16, fontweight='bold')
plt.legend()
plt.grid(True)

# Save and show the figure
plt.savefig('plots/posterior_H0.png', dpi=300, bbox_inches='tight')
plt.show()


Creating plots...


In [12]:
# --- Step 8: Additional Analysis Plots ---
print("\nCreating additional analysis plots...")

# Plot 1: Trace plots to check MCMC convergence
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Create trace plots for each parameter
for i, name in enumerate(params_names):
    if i < len(axes):
        axes[i].plot(samples[:, i], alpha=0.7, linewidth=0.5)
        axes[i].axhline(y=fiducial_values[i], color='red', linestyle='--', 
                       label=f'Fiducial: {fiducial_values[i]:.4f}')
        axes[i].set_title(f'{name} Trace')
        axes[i].set_xlabel('Sample Number')
        axes[i].set_ylabel(name)
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# Remove empty subplot
if len(params_names) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('plots/trace_plots.png', dpi=300, bbox_inches='tight')
plt.show()


Creating additional analysis plots...


In [13]:
# --- Step 9: Create Corner Plot ---
print("\nCreating corner plots...")

# 9A. corner.py plot
try:
    import corner

    fig = corner.corner(
        samples,
        labels=params_names,
        truths=fiducial_values,
        weights=weights,
        show_titles=True,
        title_kwargs={"fontsize": 12},
        quantiles=[0.16, 0.5, 0.84],
        title_fmt='.4f',
        smooth=1.0,
        bins=30,
        color='blue',
        truth_color='red',
        hist_kwargs={'density': True, 'alpha': 0.7},
        scatter_kwargs={'alpha': 0.3, 's': 10}
    )

    fig.suptitle('Galaxy-CMB Cross-Correlation: Posterior Distributions', 
                 fontsize=16, y=0.98)

    plt.savefig('plots/corner_plot_corner.png', dpi=300, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f"corner.py plot failed: {e}")

# 9B. GetDist plot
try:
    from getdist import MCSamples, plots as gd_plots

    print("Creating GetDist triangle plot...")

    samples_gd = MCSamples(samples=samples, weights=weights, names=params_names, labels=params_names)

    g = gd_plots.get_subplot_plotter()
    g.triangle_plot([samples_gd], filled=True, contour_colors=['blue'],
                    markers=dict(zip(params_names, fiducial_values)))
    
    plt.savefig('plots/corner_plot_getdist.png', dpi=300, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f"GetDist plot failed: {e}")


Creating corner plots...


Creating GetDist triangle plot...
Removed no burn in


In [14]:
# --- Step 10: Corner Plot (Parameter Correlations) Plot ---
print("Creating corner plots...")

# Try GetDist first
try:
    from getdist import MCSamples, plots

    # Create MCSamples object for GetDist
    mc_samples = MCSamples(samples=samples, weights=weights, names=params_names)

    # Create corner plot using GetDist
    g = plots.get_subplot_plotter()
    g.triangle_plot([mc_samples], filled=True,
                    markers=dict(zip(params_names, fiducial_values)),
                    colors=['blue'])
    g.export("plots/corner_plot_getdist.png")
    plt.show()

except Exception as e:
    print(f"GetDist corner plot failed: {e}")

# Use corner (corner.py) as a backup regardless
try:
    import corner

    print("Creating corner plot using `corner`")

    figure = corner.corner(
        samples,
        labels=params_names,
        truths=fiducial_values,
        weights=weights,
        show_titles=True,
        title_fmt=".4f",
        color='dodgerblue'
    )

    figure.suptitle("Corner Plot of Posterior Samples", fontsize=16)
    figure.tight_layout()
    figure.savefig("plots/corner_plot_corner.png", dpi=300)
    plt.show()

except Exception as e:
    print(f"corner.py plot failed: {e}")

Creating corner plots...
Removed no burn in


Creating corner plot using `corner`
